**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 34: 프로젝트 데이터 파이프라인 구축

## 🎯 데이터 파이프라인 구축

이 노트북에서는 프로젝트에 필요한 **데이터 파이프라인 전체**를 구축합니다.

### 파이프라인 흐름
```
데이터 수집 → 정제 → 형식 변환 → 합성 데이터 생성 → 검증 → Train/Eval 분할
```

### 사전 준비
- `32_project_planning.ipynb`에서 저장한 `project_plan.json` 필요
- OpenAI API 키 (합성 데이터 생성 시)
- HuggingFace 토큰 (데이터셋 다운로드 시)

In [ ]:
# =====================================================
# 📦 필수 패키지 설치 및 임포트
# =====================================================

# !pip install datasets openai pandas tqdm

import json
import os
import pandas as pd
from collections import Counter
from tqdm import tqdm
import re
import hashlib

print("✅ 패키지 로드 완료")

In [ ]:
# =====================================================
# 📋 프로젝트 계획서 로드
# =====================================================

PROJECT_DIR = "./my_project"  # TODO: 프로젝트 디렉토리 경로 확인

with open(f"{PROJECT_DIR}/project_plan.json", "r", encoding="utf-8") as f:
    project_plan = json.load(f)

print(f"📋 프로젝트: {project_plan['project_name']}")
print(f"📊 목표 데이터 수: {project_plan['data_strategy']['collection']['target_count']}개")
print(f"📊 데이터 형식: {project_plan['data_strategy']['format']['type']}")

---
## 1️⃣ 데이터 수집

도메인에 맞는 데이터를 수집합니다. 아래 방법 중 해당하는 것을 선택하세요.

### 수집 방법 선택
| 방법 | 장점 | 단점 | 적합한 경우 |
|------|------|------|------------|
| HuggingFace 데이터셋 | 즉시 사용 가능, 정제됨 | 도메인 제한 | 공개 데이터 있을 때 |
| 웹 크롤링 | 최신 데이터 | 노이즈 많음, 법적 이슈 | 특수 도메인 |
| 합성 데이터 (API) | 품질 제어 가능 | 비용 발생 | 데이터 부족 시 |
| 기존 데이터 변환 | 비용 없음 | 형식 변환 필요 | 자체 데이터 있을 때 |

In [ ]:
# =====================================================
# 방법 A: HuggingFace 데이터셋에서 수집
# TODO: 도메인에 맞는 데이터셋을 선택하세요
# =====================================================

from datasets import load_dataset

# TODO: 아래 데이터셋 이름과 설정을 변경하세요
# 예시 데이터셋들:
# - "heegyu/kowikitext" (한국어 위키)
# - "beomi/KoAlpaca-v1.1a" (한국어 Alpaca)
# - "nlpai-lab/kullm-v2" (한국어 LLM 학습 데이터)
# - 직접 HuggingFace에서 검색: https://huggingface.co/datasets

DATASET_NAME = ""  # TODO: 데이터셋 이름 입력
DATASET_SPLIT = "train"  # TODO: split 선택

if DATASET_NAME:
    raw_dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT)
    print(f"✅ 데이터셋 로드 완료: {len(raw_dataset)}개")
    print(f"📊 컬럼: {raw_dataset.column_names}")
    print(f"\n📝 첫 번째 예시:")
    print(raw_dataset[0])
else:
    print("⚠️ DATASET_NAME을 입력해주세요.")
    raw_dataset = None

In [ ]:
# =====================================================
# 방법 B: 로컬 파일에서 수집
# TODO: CSV, JSON, JSONL 등 로컬 데이터 로드
# =====================================================

# 예시 1: CSV 파일
# df = pd.read_csv(f"{PROJECT_DIR}/data/raw_data.csv")

# 예시 2: JSON 파일
# with open(f"{PROJECT_DIR}/data/raw_data.json", "r", encoding="utf-8") as f:
#     raw_data = json.load(f)

# 예시 3: JSONL 파일
# raw_data = []
# with open(f"{PROJECT_DIR}/data/raw_data.jsonl", "r", encoding="utf-8") as f:
#     for line in f:
#         raw_data.append(json.loads(line))

# TODO: 위 예시 중 하나를 선택하여 주석을 해제하세요
print("⚠️ 로컬 데이터 로드 코드를 작성해주세요 (필요한 경우)")

In [ ]:
# =====================================================
# 방법 C: 웹 크롤링 (기본 템플릿)
# TODO: 크롤링 대상 URL과 파싱 로직을 작성하세요
# ⚠️ robots.txt 확인, 이용약관 준수 필요
# =====================================================

# import requests
# from bs4 import BeautifulSoup

# def crawl_data(urls):
#     """URL 목록에서 데이터를 크롤링합니다."""
#     collected = []
#     for url in tqdm(urls, desc="크롤링 중"):
#         try:
#             response = requests.get(url, timeout=10)
#             soup = BeautifulSoup(response.text, "html.parser")
#             
#             # TODO: 도메인에 맞는 파싱 로직 작성
#             # question = soup.select_one(".question-title").text
#             # answer = soup.select_one(".answer-content").text
#             # collected.append({"question": question, "answer": answer})
#             pass
#         except Exception as e:
#             print(f"❌ 에러: {url} - {e}")
#     return collected

# TODO: 크롤링 실행
# urls = ["https://example.com/page1", "https://example.com/page2"]
# crawled_data = crawl_data(urls)

print("⚠️ 크롤링 코드를 작성해주세요 (필요한 경우)")

---
## 2️⃣ 데이터 정제 파이프라인

수집한 원시 데이터를 정제합니다.

### 정제 단계
1. **중복 제거**: 완전 일치 / 유사도 기반
2. **길이 필터링**: 너무 짧거나 긴 데이터 제거
3. **품질 필터링**: 의미 없는 데이터 제거
4. **언어 필터링**: 한국어 비율 확인
5. **특수문자 정리**: HTML 태그, 불필요한 기호 제거

In [ ]:
# =====================================================
# 🧹 데이터 정제 함수 모음
# =====================================================

def remove_duplicates(data, key_field):
    """
    중복 데이터를 제거합니다.
    
    Args:
        data: 데이터 리스트
        key_field: 중복 확인할 필드명
    Returns:
        중복 제거된 데이터 리스트
    """
    seen = set()
    unique_data = []
    for item in data:
        # 해시 기반 중복 확인
        text = item.get(key_field, "")
        text_hash = hashlib.md5(text.encode()).hexdigest()
        if text_hash not in seen:
            seen.add(text_hash)
            unique_data.append(item)
    
    removed = len(data) - len(unique_data)
    print(f"  중복 제거: {removed}개 제거 ({len(data)} → {len(unique_data)})")
    return unique_data


def filter_by_length(data, field, min_chars=20, max_chars=5000):
    """
    길이 기준으로 필터링합니다.
    
    Args:
        data: 데이터 리스트
        field: 길이를 확인할 필드명
        min_chars: 최소 글자 수
        max_chars: 최대 글자 수
    Returns:
        필터링된 데이터 리스트
    """
    filtered = [
        item for item in data
        if min_chars <= len(item.get(field, "")) <= max_chars
    ]
    removed = len(data) - len(filtered)
    print(f"  길이 필터: {removed}개 제거 ({len(data)} → {len(filtered)})")
    return filtered


def clean_text(text):
    """
    텍스트를 정리합니다.
    """
    # HTML 태그 제거
    text = re.sub(r"<[^>]+>", "", text)
    # 연속 공백 제거
    text = re.sub(r"\s+", " ", text)
    # 앞뒤 공백 제거
    text = text.strip()
    return text


def filter_korean_ratio(data, field, min_ratio=0.3):
    """
    한국어 비율이 일정 이상인 데이터만 남깁니다.
    
    Args:
        data: 데이터 리스트
        field: 확인할 필드명
        min_ratio: 최소 한국어 비율 (0~1)
    Returns:
        필터링된 데이터 리스트
    """
    def korean_ratio(text):
        if not text:
            return 0
        korean_chars = len(re.findall(r"[가-힣]", text))
        total_chars = len(re.findall(r"\S", text))
        return korean_chars / max(total_chars, 1)
    
    filtered = [
        item for item in data
        if korean_ratio(item.get(field, "")) >= min_ratio
    ]
    removed = len(data) - len(filtered)
    print(f"  한국어 필터: {removed}개 제거 ({len(data)} → {len(filtered)})")
    return filtered


print("✅ 정제 함수 정의 완료")

In [ ]:
# =====================================================
# 🔧 데이터 정제 실행
# TODO: 수집한 데이터를 정제하세요
# =====================================================

# TODO: raw_data에 수집한 데이터를 넣어주세요
# 형식: [{"instruction": ..., "input": ..., "output": ...}, ...]
raw_data = []  # TODO: 위에서 수집한 데이터로 교체

if raw_data:
    print(f"📊 원시 데이터: {len(raw_data)}개")
    print("\n🧹 정제 시작...")
    
    # Step 1: 텍스트 정리
    for item in raw_data:
        # TODO: 정리할 필드를 선택하세요
        for field in ["instruction", "input", "output"]:
            if field in item and item[field]:
                item[field] = clean_text(item[field])
    print("  ✅ 텍스트 정리 완료")
    
    # Step 2: 중복 제거
    # TODO: 중복 확인할 필드를 선택하세요
    cleaned_data = remove_duplicates(raw_data, key_field="instruction")
    
    # Step 3: 길이 필터링
    # TODO: min_chars, max_chars를 조정하세요
    cleaned_data = filter_by_length(cleaned_data, field="output", min_chars=20, max_chars=3000)
    
    # Step 4: 한국어 필터링
    cleaned_data = filter_korean_ratio(cleaned_data, field="output", min_ratio=0.3)
    
    print(f"\n📊 정제 결과: {len(raw_data)}개 → {len(cleaned_data)}개")
    print(f"   제거율: {(1 - len(cleaned_data)/len(raw_data))*100:.1f}%")
else:
    cleaned_data = []
    print("⚠️ raw_data를 설정해주세요.")

---
## 3️⃣ 데이터 형식 변환

정제된 데이터를 학습에 적합한 형식으로 변환합니다.

### 지원 형식

**Alpaca 형식** (Instruction Tuning 표준)
```json
{"instruction": "질문", "input": "추가 컨텍스트", "output": "답변"}
```

**ShareGPT 형식** (대화형)
```json
{"conversations": [{"from": "human", "value": "질문"}, {"from": "gpt", "value": "답변"}]}
```

In [ ]:
# =====================================================
# 🔄 데이터 형식 변환 함수
# =====================================================

def to_alpaca_format(item, instruction_field, input_field=None, output_field=None):
    """
    데이터를 Alpaca 형식으로 변환합니다.
    
    Args:
        item: 원본 데이터 딕셔너리
        instruction_field: instruction으로 사용할 필드명
        input_field: input으로 사용할 필드명 (없으면 빈 문자열)
        output_field: output으로 사용할 필드명
    Returns:
        Alpaca 형식 딕셔너리
    """
    return {
        "instruction": item.get(instruction_field, ""),
        "input": item.get(input_field, "") if input_field else "",
        "output": item.get(output_field, "") if output_field else "",
    }


def to_sharegpt_format(item, human_field, gpt_field, system_msg=None):
    """
    데이터를 ShareGPT 형식으로 변환합니다.
    
    Args:
        item: 원본 데이터 딕셔너리
        human_field: 사용자 메시지 필드명
        gpt_field: 어시스턴트 메시지 필드명
        system_msg: 시스템 메시지 (선택)
    Returns:
        ShareGPT 형식 딕셔너리
    """
    conversations = []
    if system_msg:
        conversations.append({"from": "system", "value": system_msg})
    conversations.append({"from": "human", "value": item.get(human_field, "")})
    conversations.append({"from": "gpt", "value": item.get(gpt_field, "")})
    return {"conversations": conversations}


def alpaca_to_sharegpt(alpaca_item, system_msg=None):
    """Alpaca 형식을 ShareGPT 형식으로 변환합니다."""
    human_msg = alpaca_item["instruction"]
    if alpaca_item.get("input"):
        human_msg += f"\n\n{alpaca_item['input']}"
    
    conversations = []
    if system_msg:
        conversations.append({"from": "system", "value": system_msg})
    conversations.append({"from": "human", "value": human_msg})
    conversations.append({"from": "gpt", "value": alpaca_item["output"]})
    return {"conversations": conversations}


print("✅ 변환 함수 정의 완료")

In [ ]:
# =====================================================
# 🔧 데이터 형식 변환 실행
# TODO: 데이터를 원하는 형식으로 변환하세요
# =====================================================

TARGET_FORMAT = "alpaca"  # TODO: "alpaca" 또는 "sharegpt" 선택

formatted_data = []

if cleaned_data:
    for item in cleaned_data:
        if TARGET_FORMAT == "alpaca":
            # TODO: 필드 매핑을 수정하세요
            formatted = to_alpaca_format(
                item,
                instruction_field="instruction",  # TODO: 원본 데이터의 질문 필드명
                input_field=None,                  # TODO: 원본 데이터의 입력 필드명 (없으면 None)
                output_field="output",             # TODO: 원본 데이터의 답변 필드명
            )
        elif TARGET_FORMAT == "sharegpt":
            # TODO: 필드 매핑을 수정하세요
            formatted = to_sharegpt_format(
                item,
                human_field="instruction",  # TODO: 원본 데이터의 질문 필드명
                gpt_field="output",         # TODO: 원본 데이터의 답변 필드명
                system_msg=None,            # TODO: 시스템 메시지 (선택)
            )
        formatted_data.append(formatted)
    
    print(f"✅ 형식 변환 완료: {len(formatted_data)}개 ({TARGET_FORMAT} 형식)")
    print(f"\n📝 변환 결과 예시:")
    print(json.dumps(formatted_data[0], ensure_ascii=False, indent=2))
else:
    print("⚠️ cleaned_data가 비어있습니다. 위 단계를 먼저 완료하세요.")

---
## 4️⃣ 합성 데이터 생성 (OpenAI API 활용)

데이터가 부족한 경우 **OpenAI API를 활용하여 합성 데이터를 생성**합니다.

### 합성 데이터 생성 전략
- **Seed 기반 생성**: 기존 데이터를 참고하여 유사한 새 데이터 생성
- **시나리오 기반 생성**: 도메인 시나리오를 정의하고 Q&A 쌍 생성
- **Distillation**: 큰 모델의 출력을 작은 모델의 학습 데이터로 활용

> 💡 **Hint**: `gpt-4o-mini`를 사용하면 비용을 절약하면서도 좋은 품질의 데이터를 생성할 수 있습니다.

In [ ]:
# =====================================================
# 🤖 합성 데이터 생성기
# TODO: API 키와 프롬프트를 설정하세요
# =====================================================

from openai import OpenAI
import time

# API 클라이언트 설정
client = OpenAI(
    api_key=""  # TODO: OpenAI API 키 입력 (또는 환경변수 사용)
)

# 합성 데이터 생성 모델
SYNTH_MODEL = "gpt-4o-mini"  # TODO: 모델 선택 (gpt-4o-mini 권장)


def generate_synthetic_data(
    domain_description,
    task_description,
    num_samples=10,
    seed_examples=None,
    output_format="alpaca"
):
    """
    OpenAI API를 사용하여 합성 데이터를 생성합니다.
    
    Args:
        domain_description: 도메인 설명
        task_description: 태스크 설명
        num_samples: 생성할 샘플 수
        seed_examples: 참고할 예시 데이터 (선택)
        output_format: 출력 형식 ("alpaca" 또는 "sharegpt")
    Returns:
        생성된 데이터 리스트
    """
    generated = []
    
    # 시스템 프롬프트
    system_prompt = f"""당신은 {domain_description} 도메인의 학습 데이터를 생성하는 전문가입니다.
다음 조건에 맞는 고품질 한국어 Q&A 데이터를 생성해주세요:

- 태스크: {task_description}
- 다양한 난이도와 주제를 포함
- 실제 사용자가 물어볼 법한 자연스러운 질문
- 정확하고 도움이 되는 상세한 답변
- 한국어로 작성

출력 형식 (JSON):
{{
    "instruction": "질문",
    "input": "추가 컨텍스트 (없으면 빈 문자열)",
    "output": "상세한 답변"
}}"""

    # 예시가 있으면 포함
    examples_text = ""
    if seed_examples:
        examples_text = "\n\n참고 예시:\n"
        for ex in seed_examples[:3]:
            examples_text += json.dumps(ex, ensure_ascii=False) + "\n"
    
    # 배치로 생성 (한 번에 5개씩)
    batch_size = 5
    for i in tqdm(range(0, num_samples, batch_size), desc="합성 데이터 생성"):
        current_batch = min(batch_size, num_samples - i)
        
        user_prompt = f"""{examples_text}
위 형식에 맞는 새로운 데이터를 {current_batch}개 생성해주세요.
각 데이터는 서로 다른 주제/상황이어야 합니다.
JSON 배열로 출력해주세요."""
        
        try:
            response = client.chat.completions.create(
                model=SYNTH_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.8,
                max_tokens=4000,
            )
            
            # JSON 파싱
            content = response.choices[0].message.content
            # JSON 배열 추출
            json_match = re.search(r"\[.*\]", content, re.DOTALL)
            if json_match:
                batch_data = json.loads(json_match.group())
                generated.extend(batch_data)
            
            time.sleep(1)  # Rate limit 대응
            
        except Exception as e:
            print(f"❌ 생성 오류: {e}")
    
    print(f"\n✅ 합성 데이터 생성 완료: {len(generated)}개")
    return generated


print("✅ 합성 데이터 생성 함수 정의 완료")

In [ ]:
# =====================================================
# 🔧 합성 데이터 생성 실행
# TODO: 도메인과 태스크를 설정하세요
# =====================================================

# TODO: 아래 변수를 프로젝트에 맞게 수정하세요
DOMAIN_DESC = ""       # TODO: 도메인 설명 (예: "의료/건강 상담")
TASK_DESC = ""         # TODO: 태스크 설명 (예: "환자의 증상을 듣고 가능한 질환과 대처법을 안내")
NUM_SYNTHETIC = 50     # TODO: 생성할 데이터 수

if DOMAIN_DESC and TASK_DESC:
    # 기존 데이터에서 시드 예시 선택
    seed_examples = formatted_data[:5] if formatted_data else None
    
    synthetic_data = generate_synthetic_data(
        domain_description=DOMAIN_DESC,
        task_description=TASK_DESC,
        num_samples=NUM_SYNTHETIC,
        seed_examples=seed_examples,
    )
    
    # 기존 데이터와 합치기
    all_data = formatted_data + synthetic_data
    print(f"\n📊 전체 데이터: {len(formatted_data)} (원본) + {len(synthetic_data)} (합성) = {len(all_data)}개")
else:
    synthetic_data = []
    all_data = formatted_data
    print("⚠️ DOMAIN_DESC와 TASK_DESC를 설정해주세요.")
    print(f"📊 현재 데이터: {len(all_data)}개 (합성 데이터 미생성)")

---
## 5️⃣ 데이터 검증 및 통계

최종 데이터의 품질을 검증하고 통계를 확인합니다.

In [ ]:
# =====================================================
# 📊 데이터 통계 분석
# =====================================================

def analyze_dataset(data, format_type="alpaca"):
    """
    데이터셋의 통계를 분석합니다.
    
    Args:
        data: 데이터 리스트
        format_type: "alpaca" 또는 "sharegpt"
    """
    if not data:
        print("⚠️ 데이터가 비어있습니다.")
        return
    
    print(f"📊 데이터셋 통계")
    print(f"{'='*50}")
    print(f"전체 데이터 수: {len(data)}개")
    
    if format_type == "alpaca":
        # 길이 통계
        inst_lens = [len(d.get("instruction", "")) for d in data]
        out_lens = [len(d.get("output", "")) for d in data]
        input_lens = [len(d.get("input", "")) for d in data]
        
        print(f"\n📝 Instruction 길이:")
        print(f"   평균: {sum(inst_lens)/len(inst_lens):.0f}자")
        print(f"   최소: {min(inst_lens)}자 / 최대: {max(inst_lens)}자")
        
        print(f"\n📝 Output 길이:")
        print(f"   평균: {sum(out_lens)/len(out_lens):.0f}자")
        print(f"   최소: {min(out_lens)}자 / 최대: {max(out_lens)}자")
        
        has_input = sum(1 for d in data if d.get("input", ""))
        print(f"\n📝 Input 필드 사용: {has_input}/{len(data)} ({has_input/len(data)*100:.0f}%)")
        
    elif format_type == "sharegpt":
        turn_counts = [len(d.get("conversations", [])) for d in data]
        print(f"\n📝 대화 턴 수:")
        print(f"   평균: {sum(turn_counts)/len(turn_counts):.1f}턴")
        print(f"   최소: {min(turn_counts)}턴 / 최대: {max(turn_counts)}턴")
    
    # 샘플 출력
    print(f"\n📝 랜덤 샘플 (3개):")
    import random
    samples = random.sample(data, min(3, len(data)))
    for i, sample in enumerate(samples):
        print(f"\n--- 샘플 {i+1} ---")
        print(json.dumps(sample, ensure_ascii=False, indent=2)[:500])


# 통계 분석 실행
if all_data:
    analyze_dataset(all_data, format_type=TARGET_FORMAT)
else:
    print("⚠️ 분석할 데이터가 없습니다.")

In [ ]:
# =====================================================
# ✅ 데이터 품질 검증
# =====================================================

def validate_dataset(data, format_type="alpaca"):
    """
    데이터셋의 품질을 검증합니다.
    
    Args:
        data: 데이터 리스트
        format_type: "alpaca" 또는 "sharegpt"
    Returns:
        유효한 데이터 리스트
    """
    valid = []
    issues = []
    
    for i, item in enumerate(data):
        if format_type == "alpaca":
            # 필수 필드 확인
            if not item.get("instruction"):
                issues.append(f"[{i}] instruction이 비어있음")
                continue
            if not item.get("output"):
                issues.append(f"[{i}] output이 비어있음")
                continue
            # output이 너무 짧은 경우
            if len(item["output"]) < 10:
                issues.append(f"[{i}] output이 너무 짧음 ({len(item['output'])}자)")
                continue
                
        elif format_type == "sharegpt":
            convs = item.get("conversations", [])
            if len(convs) < 2:
                issues.append(f"[{i}] 대화 턴이 부족함 ({len(convs)}턴)")
                continue
        
        valid.append(item)
    
    print(f"✅ 검증 결과: {len(valid)}/{len(data)}개 유효")
    if issues:
        print(f"\n⚠️ 이슈 발견 ({len(issues)}개):")
        for issue in issues[:10]:  # 처음 10개만 출력
            print(f"   - {issue}")
        if len(issues) > 10:
            print(f"   ... 그 외 {len(issues)-10}개")
    
    return valid


# 검증 실행
if all_data:
    valid_data = validate_dataset(all_data, format_type=TARGET_FORMAT)
else:
    valid_data = []
    print("⚠️ 검증할 데이터가 없습니다.")

---
## 6️⃣ Train/Eval 분할

검증된 데이터를 **학습용(Train)**과 **평가용(Eval)**으로 분할합니다.

In [ ]:
# =====================================================
# ✂️ Train/Eval 분할 및 저장
# TODO: 분할 비율을 조정하세요
# =====================================================

import random

TRAIN_RATIO = 0.9  # TODO: 학습 데이터 비율 (기본 90%)
RANDOM_SEED = 42

if valid_data:
    # 셔플
    random.seed(RANDOM_SEED)
    shuffled = valid_data.copy()
    random.shuffle(shuffled)
    
    # 분할
    split_idx = int(len(shuffled) * TRAIN_RATIO)
    train_data = shuffled[:split_idx]
    eval_data = shuffled[split_idx:]
    
    print(f"📊 데이터 분할 결과:")
    print(f"   Train: {len(train_data)}개 ({len(train_data)/len(shuffled)*100:.0f}%)")
    print(f"   Eval:  {len(eval_data)}개 ({len(eval_data)/len(shuffled)*100:.0f}%)")
    
    # 저장
    data_dir = f"{PROJECT_DIR}/data"
    os.makedirs(data_dir, exist_ok=True)
    
    # Train 데이터 저장
    train_path = f"{data_dir}/train.json"
    with open(train_path, "w", encoding="utf-8") as f:
        json.dump(train_data, f, ensure_ascii=False, indent=2)
    
    # Eval 데이터 저장
    eval_path = f"{data_dir}/eval.json"
    with open(eval_path, "w", encoding="utf-8") as f:
        json.dump(eval_data, f, ensure_ascii=False, indent=2)
    
    # 전체 데이터도 저장
    all_path = f"{data_dir}/all_data.json"
    with open(all_path, "w", encoding="utf-8") as f:
        json.dump(valid_data, f, ensure_ascii=False, indent=2)
    
    print(f"\n💾 저장 완료:")
    print(f"   - {train_path}")
    print(f"   - {eval_path}")
    print(f"   - {all_path}")
else:
    print("⚠️ 분할할 데이터가 없습니다.")

---
## 📝 정리

### 이번 세션에서 완료한 것
- ✅ 데이터 수집 (HuggingFace / 크롤링 / 로컬 파일)
- ✅ 데이터 정제 (중복 제거, 길이 필터, 한국어 필터)
- ✅ 데이터 형식 변환 (Alpaca / ShareGPT)
- ✅ 합성 데이터 생성 (OpenAI API)
- ✅ 데이터 검증 및 통계 분석
- ✅ Train/Eval 분할 및 저장

### 산출물
- 📁 `my_project/data/train.json` - 학습용 데이터
- 📁 `my_project/data/eval.json` - 평가용 데이터
- 📁 `my_project/data/all_data.json` - 전체 데이터

### 다음 세션 준비사항
- 📌 데이터 파일이 올바르게 저장되었는지 확인
- 📌 데이터 수가 최소 300개 이상인지 확인
- 📌 데이터 형식이 올바른지 JSON 구조 확인

### 다음 노트북
👉 **34_project_training.ipynb**: 프로젝트 모델 학습 수행